# **Model Building**

In [ ]:
import pandas as pd
data = pd.read_csv("/content/drive/MyDrive/datascience/Mini Projects/Diamond Dynamics: Price Prediction and Market Segmentation/cleaned_data.csv")
data.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.50,55.0,326.0,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.80,61.0,326.0,3.89,3.84,2.31
2,0.23,Good,E,VS1,58.75,63.5,327.0,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.40,58.0,334.0,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.30,58.0,335.0,4.34,4.35,2.75


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder,StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
from sklearn.model_selection import cross_val_score, KFold
import numpy as np

In [ ]:
# Define columns
ordinal_cols = ['cut', 'color', 'clarity']
scaling_cols = ['depth', 'table', 'x', 'y', 'z']
pos_skewed_cols = ['carat']

# Define transformations
preprocessor = ColumnTransformer(
    transformers=[
        # Using 'handle_unknown' in case new data has categories not seen in training
        ('ore', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ordinal_cols),
        ('scaler', StandardScaler(), scaling_cols),
        ('log', FunctionTransformer(np.log1p, validate=True), pos_skewed_cols)
    ]
)


# Splitting data ('price' is the target variable)
X = data.drop(columns=["price"])
y = data["price"]

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "XGBoost Regressor": XGBRegressor(),
    "KNN Regressor": KNeighborsRegressor(n_neighbors=5)
}
# Train and evaluate models
results = {}

for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    # Train the model
    pipeline.fit(X_train, y_train)

    # Make predictions
    y_test_pred = pipeline.predict(X_test)

    # Compute evaluation metrics
    train_accuracy = pipeline.score(X_train, y_train)
    test_accuracy = pipeline.score(X_test, y_test)
    mae = mean_absolute_error(y_test, y_test_pred)
    mse = mean_squared_error(y_test, y_test_pred)
    rmse = root_mean_squared_error(y_test, y_test_pred)
    r2 = r2_score(y_test, y_test_pred)

    results[name] = {
        "Train Accuracy": train_accuracy,
        "Test Accuracy": test_accuracy,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2 Score": r2
    }

# Convert results to DataFrame
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values(by="R2 Score", ascending=False)

results_df

,Train Accuracy,Test Accuracy,MAE,MSE,RMSE,R2 Score
XGBoost Regressor,0.992916,0.987791,212.913080,1.403340e+05,374.611834,0.987791
Random Forest Regressor,0.998241,0.987472,204.630320,1.440008e+05,379.474319,0.987472
Decision Tree Regressor,0.999998,0.976043,273.078095,2.753609e+05,524.748399,0.976043
KNN Regressor,0.981111,0.970629,332.070898,3.375814e+05,581.017534,0.970629
Linear Regression,0.893437,0.891380,821.163536,1.248470e+06,1117.349485,0.891380


# **Crossvalidation**

In [ ]:
from sklearn.model_selection import cross_validate, KFold
import pandas as pd

# Define K-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

model_results = {}

for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    # Define metrics as strings that scikit-learn understands
    metrics = {
        'MAE': 'neg_mean_absolute_error',
        'RMSE': 'neg_root_mean_squared_error',
        'MSE': 'neg_mean_squared_error',
        'R2': 'r2'
    }

    # Run cross_validate on the FULL X and y
    # return_train_score=True automatically calculates training accuracy for every fold
    cv_results = cross_validate(
        pipeline, X, y,
        cv=kf,
        scoring=metrics,
        return_train_score=True
    )

    # Extract results (scikit-learn adds 'train_' and 'test_' prefixes)
    model_results[name] = {
        "Train R2 (Accuracy)": cv_results['train_R2'].mean(),
        "Test R2 (Accuracy)": cv_results['test_R2'].mean(),
        "MAE": -cv_results['train_MAE'].mean(),
        "MAE": -cv_results['test_MAE'].mean(),
        "RMSE": -cv_results['test_RMSE'].mean()
    }

# Convert results to DataFrame
results_df = pd.DataFrame(model_results).T
results_df = results_df.sort_values(by="Test R2 (Accuracy)", ascending=False)

results_df

,Train R2 (Accuracy),Test R2 (Accuracy),MAE,RMSE
XGBoost Regressor,0.992673,0.988129,211.919161,374.402139
Random Forest Regressor,0.998282,0.987867,202.978770,378.476526
Decision Tree Regressor,0.999997,0.977344,270.516044,517.198149
KNN Regressor,0.982081,0.972400,324.351707,570.768348
Linear Regression,0.892911,0.892462,822.373017,1126.834644


# **ANN Regression**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Transform the raw data into a numerical NumPy array using your preprocessor
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# Convert the numerical arrays to PyTorch Tensors
X_train_tensor = torch.tensor(X_train_transformed, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

X_test_tensor = torch.tensor(X_test_transformed, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)


In [ ]:
class DiamondPriceANN(nn.Module):
    def __init__(self, input_size):
        super(DiamondPriceANN, self).__init__()
        self.layer1 = nn.Linear(input_size, 64)
        self.layer2 = nn.Linear(64, 32)
        self.layer3 = nn.Linear(32, 16)
        self.output = nn.Linear(16, 1) # Single output for regression
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.relu(self.layer3(x))
        x = self.output(x) # No activation for the final regression output
        return x

# Initialize model
model = DiamondPriceANN(X_train.shape[1])
criterion = nn.MSELoss() # Loss function for regression
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 1000
for epoch in range(epochs):
    # Forward pass
    predictions = model(X_train_tensor)
    loss = criterion(predictions, y_train_tensor)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')


Epoch [100/1000], Loss: 25160370.0000
Epoch [200/1000], Loss: 9638355.0000
Epoch [300/1000], Loss: 4425757.5000
Epoch [400/1000], Loss: 2570357.5000
Epoch [500/1000], Loss: 1578573.0000
Epoch [600/1000], Loss: 1147808.5000
Epoch [700/1000], Loss: 957610.8750
Epoch [800/1000], Loss: 891012.7500
Epoch [900/1000], Loss: 871715.6875
Epoch [1000/1000], Loss: 864135.1250


In [ ]:
# Ensure the model is in eval mode and get predictions
model.eval()
with torch.no_grad():
    y_train_pred = model(X_train_tensor).numpy()
    y_test_pred = model(X_test_tensor).numpy()

# Compute evaluation metrics using the original y_train and y_test
train_accuracy = r2_score(y_train, y_train_pred)
test_accuracy = r2_score(y_test, y_test_pred)

mae = mean_absolute_error(y_test, y_test_pred)
mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_test_pred)

# Print the results
print(f"ANN Train Accuracy (R²): {train_accuracy:.4f}")
print(f"ANN Test Accuracy (R²): {test_accuracy:.4f}")
print(f"MAE: {mae:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R2 Score: {r2:.4f}")

ANN Train Accuracy (R²): 0.9277
ANN Test Accuracy (R²): 0.9262
MAE: 599.1057
MSE: 848752.2774
RMSE: 921.2775
R2 Score: 0.9262


# **Hyperparameter Tuning for XGBoost using RandomizedSearchCV**

In [ ]:
from xgboost import XGBRegressor # Import this!
from sklearn.model_selection import RandomizedSearchCV

# Update the grid with 'regressor__' prefix
xgb_param_grid = {
    'regressor__n_estimators': [500, 1000, 1500],
    'regressor__max_depth': [3, 5, 6, 8],
    'regressor__learning_rate': [0.01, 0.05, 0.1],
    'regressor__min_child_weight': [1, 2],
    'regressor__gamma': [0, 0.05]
}

# Use XGBRegressor for price prediction
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(eval_metric='rmse')) # Use Regressor here
])

# Perform RandomizedSearchCV
# Note: cv=25 is very high and will be slow; cv=5 is usually enough
xgb_search = RandomizedSearchCV(xgb_pipeline, xgb_param_grid, cv=5, verbose=1, n_jobs=-1,n_iter=10,scoring='r2')

# Fit on training data
xgb_search.fit(X, y)

print("Best Parameters:", xgb_search.best_params_)


Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best Parameters: {'regressor__n_estimators': 1500, 'regressor__min_child_weight': 1, 'regressor__max_depth': 3, 'regressor__learning_rate': 0.1, 'regressor__gamma': 0.05}


In [ ]:
# Extract the actual best model (the pipeline)
best_model = xgb_search.best_estimator_

# R^2 score on training data
train_r2 = best_model.score(X_train, y_train)

# R^2 score on test data (The true test)
test_r2 = best_model.score(X_test, y_test)

print(f"Train R² Score: {train_r2:.4f}")
print(f"Test R² Score: {test_r2:.4f}")
# Note: Use xgb_search (the name of your search object) to get the best_score_
print(f"Best CV Score: {xgb_search.best_score_:.4f}")


Train R² Score: 0.9895
Test R² Score: 0.9894
Best CV Score: 0.7170


In [ ]:
from sklearn.metrics import mean_absolute_error

y_pred = best_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)

print(f"On average, the model's price prediction is off by: {mae:.4f}")


On average, the model's price prediction is off by: 206.8335


# **Model into pickle file**

In [ ]:
import pickle
# To be safe, save the entire Pipeline (Preprocessor + Model)
with open('Diamond_price.pkl', 'wb') as file:
    pickle.dump(xgb_search.best_estimator_, file)

print("Full pipeline saved! It's ready for raw data input.")


Full pipeline saved! It's ready for raw data input.
